# PubMed Keyword Extractor
Searches PubMed for a term and extracts keywords and MeSH terms from top articles.

**Instructions:** Fill in the configuration cell below, then run all cells (Runtime → Run all).

In [ ]:
QUERY = "food security"
LIMIT = 100
YEARS = 5
OUTPUT_PREFIX = "pubmed_results"

In [ ]:
import time
import csv
import json
from datetime import datetime, timedelta
from collections import Counter
from xml.etree import ElementTree as ET
import urllib.request
import urllib.parse
import urllib.error

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def search_pubmed(query, limit=100, min_date=None, max_date=None):
    full_query = query
    if min_date and max_date:
        full_query = f'({query}) AND ("{min_date}"[Date - Publication] : "{max_date}"[Date - Publication])'

    params = {
        "db": "pubmed",
        "term": full_query,
        "retmax": limit,
        "retmode": "json",
        "sort": "relevance",
    }

    url = f"{BASE_URL}/esearch.fcgi?{urllib.parse.urlencode(params)}"
    print(f"Searching PubMed for: {full_query}")

    try:
        with urllib.request.urlopen(url, timeout=30) as response:
            data = json.loads(response.read().decode())
    except urllib.error.URLError as e:
        print(f"Error connecting to PubMed: {e}")
        return []

    result = data.get("esearchresult", {})
    pmids = result.get("idlist", [])
    print(f"Found {result.get('count', 0)} total results, retrieved {len(pmids)} articles")
    return pmids


def fetch_citation_counts(pmids, batch_size=50):
    citation_counts = {}
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        print(f"Fetching citation counts {i + 1}-{min(i + batch_size, len(pmids))}...")
        params = {"dbfrom": "pubmed", "linkname": "pubmed_pubmed_citedin", "retmode": "xml"}
        id_params = "&".join(f"id={pmid}" for pmid in batch)
        url = f"{BASE_URL}/elink.fcgi?{urllib.parse.urlencode(params)}&{id_params}"
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                root = ET.fromstring(response.read().decode())
        except Exception as e:
            print(f"Error: {e}")
            for pmid in batch:
                citation_counts[pmid] = 0
            continue
        for linkset in root.findall(".//LinkSet"):
            id_elem = linkset.find(".//IdList/Id")
            if id_elem is None:
                continue
            pmid = id_elem.text
            count = 0
            for linksetdb in linkset.findall(".//LinkSetDb"):
                linkname_elem = linksetdb.find("LinkName")
                if linkname_elem is not None and linkname_elem.text == "pubmed_pubmed_citedin":
                    count = len(linksetdb.findall("Link"))
                    break
            citation_counts[pmid] = count
        for pmid in batch:
            if pmid not in citation_counts:
                citation_counts[pmid] = 0
        time.sleep(0.34)
    return citation_counts


def fetch_article_details(pmids, batch_size=50):
    articles = []
    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        print(f"Fetching article details {i + 1}-{min(i + batch_size, len(pmids))}...")
        params = {"db": "pubmed", "id": ",".join(batch), "retmode": "xml"}
        url = f"{BASE_URL}/efetch.fcgi?{urllib.parse.urlencode(params)}"
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                root = ET.fromstring(response.read().decode())
        except Exception as e:
            print(f"Error: {e}")
            continue
        for article_elem in root.findall(".//PubmedArticle"):
            article = parse_article(article_elem)
            if article:
                articles.append(article)
        time.sleep(0.34)
    return articles


def parse_article(article_elem):
    article = {
        "pmid": "", "title": "", "authors": [], "journal": "",
        "pub_date": "", "year": "", "doi": "", "pubmed_url": "",
        "keywords": [], "mesh_terms": [], "abstract": "", "citations": 0,
    }
    pmid_elem = article_elem.find(".//PMID")
    if pmid_elem is not None:
        article["pmid"] = pmid_elem.text
        article["pubmed_url"] = f"https://pubmed.ncbi.nlm.nih.gov/{pmid_elem.text}/"
    title_elem = article_elem.find(".//ArticleTitle")
    if title_elem is not None:
        article["title"] = "".join(title_elem.itertext())
    for author in article_elem.findall(".//Author"):
        last = author.find("LastName")
        first = author.find("ForeName")
        if last is not None:
            article["authors"].append(f"{first.text} {last.text}" if first is not None else last.text)
    journal_elem = article_elem.find(".//Journal/Title")
    if journal_elem is not None:
        article["journal"] = journal_elem.text
    pub_date = article_elem.find(".//PubDate")
    if pub_date is not None:
        parts = [e.text for e in [pub_date.find("Year"), pub_date.find("Month"), pub_date.find("Day")] if e is not None]
        article["pub_date"] = " ".join(parts)
        year = pub_date.find("Year")
        if year is not None:
            article["year"] = year.text
    for id_elem in article_elem.findall(".//ArticleId"):
        if id_elem.get("IdType") == "doi":
            article["doi"] = id_elem.text
            break
    article["keywords"] = [k.text.strip() for k in article_elem.findall(".//Keyword") if k.text]
    article["mesh_terms"] = [m.text.strip() for m in article_elem.findall(".//MeshHeading/DescriptorName") if m.text]
    abstract_parts = []
    for t in article_elem.findall(".//AbstractText"):
        if t.text:
            label = t.get("Label", "")
            text = "".join(t.itertext())
            abstract_parts.append(f"{label}: {text}" if label else text)
    article["abstract"] = " ".join(abstract_parts)
    return article


def analyze_keywords(articles):
    all_keywords = []
    all_mesh = []
    for article in articles:
        all_keywords.extend(article.get("keywords", []))
        all_mesh.extend(article.get("mesh_terms", []))
    return {
        "total_articles": len(articles),
        "articles_with_keywords": sum(1 for a in articles if a.get("keywords")),
        "articles_with_mesh": sum(1 for a in articles if a.get("mesh_terms")),
        "keyword_frequency": Counter(all_keywords).most_common(),
        "mesh_frequency": Counter(all_mesh).most_common(),
        "combined_frequency": Counter(all_keywords + all_mesh).most_common(),
    }


def save_results(articles, analysis, output_prefix):
    keywords_file = f"{output_prefix}_keywords.csv"
    with open(keywords_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Keyword/MeSH Term", "Frequency", "Type"])
        for keyword, count in analysis["keyword_frequency"]:
            writer.writerow([keyword, count, "Author Keyword"])
        for mesh, count in analysis["mesh_frequency"]:
            writer.writerow([mesh, count, "MeSH Term"])

    articles_file = f"{output_prefix}_articles.csv"
    with open(articles_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Rank", "PMID", "Title", "Authors", "Journal", "Year",
                         "Publication Date", "DOI", "DOI URL", "PubMed URL",
                         "Citations", "Keywords", "MeSH Terms", "Abstract"])
        for rank, article in enumerate(articles, start=1):
            doi = article.get("doi", "")
            writer.writerow([
                rank, article.get("pmid", ""), article.get("title", ""),
                "; ".join(article.get("authors", [])), article.get("journal", ""),
                article.get("year", ""), article.get("pub_date", ""),
                doi, f"https://doi.org/{doi}" if doi else "",
                article.get("pubmed_url", ""), article.get("citations", 0),
                "; ".join(article.get("keywords", [])),
                "; ".join(article.get("mesh_terms", [])),
                article.get("abstract", ""),
            ])

    return keywords_file, articles_file

In [ ]:
min_date, max_date = None, None
if YEARS:
    today = datetime.now()
    min_date = (today - timedelta(days=YEARS * 365)).strftime("%Y/%m/%d")
    max_date = today.strftime("%Y/%m/%d")

pmids = search_pubmed(QUERY, limit=LIMIT, min_date=min_date, max_date=max_date)

if pmids:
    articles = fetch_article_details(pmids)
    citation_counts = fetch_citation_counts(pmids)
    for article in articles:
        article["citations"] = citation_counts.get(article["pmid"], 0)
    articles_by_pmid = {a["pmid"]: a for a in articles if a["pmid"]}
    articles = [articles_by_pmid[pmid] for pmid in pmids if pmid in articles_by_pmid]
    analysis = analyze_keywords(articles)

    print(f"\nTotal articles: {analysis['total_articles']}")
    print(f"Articles with author keywords: {analysis['articles_with_keywords']}")
    print(f"Articles with MeSH terms: {analysis['articles_with_mesh']}")
    print("\nTop 20 Author Keywords:")
    for keyword, count in analysis["keyword_frequency"][:20]:
        print(f"  {count:4d} | {keyword}")
    print("\nTop 20 MeSH Terms:")
    for mesh, count in analysis["mesh_frequency"][:20]:
        print(f"  {count:4d} | {mesh}")
else:
    print("No results found.")

In [ ]:
from google.colab import files

if pmids and articles:
    keywords_file, articles_file = save_results(articles, analysis, OUTPUT_PREFIX)
    files.download(keywords_file)
    files.download(articles_file)
    print(f"Downloaded: {keywords_file}, {articles_file}")